# 02 — Feature Engineering

Prototype các feature cho time series forecasting + anomaly detection,
trước khi port sang PySpark Glue Job (`glue_jobs/silver_to_ml_features.py`).

**Feature categories**:
1. Lag features: `aqi_lag_{1,3,6,12,24}h`
2. Rolling stats: mean / std / min / max theo cửa sổ 3h, 24h, 7d
3. Time features: hour, day_of_week, month, is_weekend, season
4. Weather features (đã có): temp, humidity, pressure, wind
5. City encoding (one-hot hoặc target encoding)
6. Rate-of-change features: `aqi_diff_1h`, `aqi_pct_change_24h`

**Validation**: Mutual Information giữa features và target `aqi_target_t+24h`.

In [ ]:
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

## 1. Load & Clean (reuse from notebook 01)

In [ ]:
CSV_PATH = Path('../raw_data_csv/historical_air_quality_2021_en.csv')

def load_clean():
    raw = pd.read_csv(CSV_PATH, na_values=['-', '', '#NAME?'], low_memory=False)
    rename_map = {
        'Station ID': 'station_id', 'AQI index': 'aqi',
        'Location': 'geo', 'Station name': 'station_name',
        'Dominent pollutant': 'dominant_pollutant',
        'CO': 'co', 'Humidity': 'humidity', 'NO2': 'no2', 'O3': 'o3',
        'Pressure': 'pressure', 'PM10': 'pm10', 'PM2.5': 'pm25', 'SO2': 'so2',
        'Temperature': 'temperature', 'Wind': 'wind', 'Data Time S': 'measured_at',
    }
    df = raw.rename(columns=rename_map)
    df['pressure'] = df['pressure'].astype(str).str.replace(',', '')
    num_cols = ['aqi', 'co', 'humidity', 'no2', 'o3', 'pressure',
                'pm10', 'pm25', 'so2', 'temperature', 'wind']
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df['measured_at'] = pd.to_datetime(df['measured_at'], errors='coerce')

    def extract_city(name):
        if pd.isna(name): return 'unknown'
        name = str(name).lower()
        if 'hanoi' in name or 'hà nội' in name: return 'ha-noi'
        if 'ho chi minh' in name or 'hồ chí minh' in name: return 'ho-chi-minh-city'
        if 'da nang' in name or 'đà nẵng' in name: return 'da-nang'
        if 'gia lai' in name or 'pleiku' in name: return 'gia-lai'
        if 'cao bằng' in name or 'cao bang' in name: return 'cao-bang'
        return 'other'
    df['queried_city'] = df['station_name'].apply(extract_city)
    return df

df = load_clean()
target_cities = ['ha-noi', 'ho-chi-minh-city', 'da-nang', 'gia-lai', 'cao-bang']
df = df[df['queried_city'].isin(target_cities)].copy()

df = (df.dropna(subset=['measured_at', 'aqi', 'station_id'])
        .sort_values(['station_id', 'measured_at']))

df_h = (df.groupby(['queried_city', pd.Grouper(key='measured_at', freq='h')])
          .agg(aqi=('aqi', 'mean'),
               pm25=('pm25', 'mean'),
               pm10=('pm10', 'mean'),
               temperature=('temperature', 'mean'),
               humidity=('humidity', 'mean'),
               pressure=('pressure', 'mean'),
               wind=('wind', 'mean'))
          .reset_index())

print(f'Hourly records: {len(df_h):,}')
print(f'Per city: {df_h.groupby("queried_city").size().to_dict()}')
df_h.head()

## 2. Build Features (per city)

In [ ]:
def build_features(df_city: pd.DataFrame, forecast_horizon_h: int = 24) -> pd.DataFrame:
    """
    df_city: single-city hourly dataframe sorted by measured_at.
    Returns: df with lag + rolling + time features and target column `aqi_target`.
    """
    d = df_city.set_index('measured_at').sort_index().copy()
    d = d.asfreq('h')
    for c in ['aqi', 'pm25', 'pm10', 'temperature', 'humidity', 'pressure', 'wind']:
        d[c] = d[c].interpolate(method='time', limit=6)

    for lag in [1, 3, 6, 12, 24]:
        d[f'aqi_lag_{lag}h']  = d['aqi'].shift(lag)
        d[f'pm25_lag_{lag}h'] = d['pm25'].shift(lag)

    for window, label in [(3, '3h'), (24, '24h'), (24*7, '7d')]:
        d[f'aqi_roll_mean_{label}'] = d['aqi'].shift(1).rolling(window).mean()
        d[f'aqi_roll_std_{label}']  = d['aqi'].shift(1).rolling(window).std()
        d[f'aqi_roll_min_{label}']  = d['aqi'].shift(1).rolling(window).min()
        d[f'aqi_roll_max_{label}']  = d['aqi'].shift(1).rolling(window).max()

    d['aqi_diff_1h']        = d['aqi'].diff(1).shift(1)
    d['aqi_pct_change_24h'] = d['aqi'].pct_change(24).shift(1)

    d['hour']       = d.index.hour
    d['dow']        = d.index.dayofweek
    d['month']      = d.index.month
    d['is_weekend'] = (d['dow'] >= 5).astype(int)
    # Vietnam dry season ~ Nov-Apr (cao AQI), wet season May-Oct
    d['is_dry_season'] = d['month'].isin([11, 12, 1, 2, 3, 4]).astype(int)

    d['hour_sin']  = np.sin(2 * np.pi * d['hour'] / 24)
    d['hour_cos']  = np.cos(2 * np.pi * d['hour'] / 24)
    d['month_sin'] = np.sin(2 * np.pi * d['month'] / 12)
    d['month_cos'] = np.cos(2 * np.pi * d['month'] / 12)

    d['aqi_target'] = d['aqi'].shift(-forecast_horizon_h)

    return d.reset_index()


feats_all = []
for city in target_cities:
    sub = df_h[df_h['queried_city'] == city].copy()
    if len(sub) < 50:
        continue
    f = build_features(sub, forecast_horizon_h=24)
    f['queried_city'] = city
    feats_all.append(f)

feats = pd.concat(feats_all, ignore_index=True)
print(f'Feature rows: {len(feats):,}')
print(f'Columns ({feats.shape[1]}): {list(feats.columns)}')
feats.head()

## 3. Feature Quality Check

In [ ]:
missing_by_feat = feats.isna().mean().mul(100).round(2).sort_values(ascending=False)
print('Missing % per column:')
print(missing_by_feat.to_string())

feats_clean = feats.dropna(subset=['aqi_target']).copy()
feature_cols = [c for c in feats_clean.columns
                if c not in ('measured_at', 'queried_city', 'aqi_target')]
feats_clean = feats_clean.dropna(subset=feature_cols)
print(f'\nAfter dropna: {len(feats_clean):,} rows ({len(feats_clean)/len(feats)*100:.1f}% of original)')

## 4. Mutual Information — Feature Importance Baseline

In [ ]:
from sklearn.feature_selection import mutual_info_regression

X = feats_clean[feature_cols]
y = feats_clean['aqi_target']

X_sample = X.sample(min(5000, len(X)), random_state=42)
y_sample = y.loc[X_sample.index]

mi = mutual_info_regression(X_sample, y_sample, random_state=42, n_neighbors=5)
mi_df = (pd.DataFrame({'feature': feature_cols, 'mutual_info': mi})
           .sort_values('mutual_info', ascending=False)
           .reset_index(drop=True))

fig, ax = plt.subplots(figsize=(10, max(5, len(mi_df) * 0.25)))
sns.barplot(data=mi_df.head(25), x='mutual_info', y='feature',
            palette='viridis', ax=ax)
ax.set_title('Top-25 Features by Mutual Information with AQI(t+24h)')
plt.tight_layout(); plt.show()

mi_df.head(25)

**Insights từ MI**:
- **Rolling means** (đặc biệt `aqi_roll_mean_24h`, `aqi_roll_mean_7d`) thường rank top — lịch sử gần là signal mạnh nhất
- **Lag features** gần (1h, 3h) có giá trị cao cho short-term
- **Cyclical time features** (hour_sin/cos, month_sin/cos) capture seasonality tốt hơn raw hour/month
- **Weather** thường rank thấp hơn — contribution nhỏ nhưng not zero

## 5. Feature Distributions Sanity Check

In [ ]:
lag_cols = [c for c in feats_clean.columns if c.startswith('aqi_lag_')]
roll_cols = [c for c in feats_clean.columns if c.startswith('aqi_roll_mean_')]

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
feats_clean[lag_cols].plot.box(ax=axes[0])
axes[0].set_title('Lag Features — Distribution')
axes[0].tick_params(axis='x', rotation=30)

feats_clean[roll_cols + ['aqi']].plot.box(ax=axes[1])
axes[1].set_title('Rolling Mean Features — Distribution')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout(); plt.show()

## 6. Save Processed Features (cho notebook 03 và 04)

In [ ]:
OUT_DIR = Path('./artifacts'); OUT_DIR.mkdir(exist_ok=True)
OUT_PATH = OUT_DIR / 'features_2021.parquet'

feats_clean.to_parquet(OUT_PATH, index=False)
print(f'Saved {len(feats_clean):,} rows to {OUT_PATH}')
print(f'File size: {OUT_PATH.stat().st_size / 1024:.1f} KB')

mi_df.to_csv(OUT_DIR / 'feature_importance_mi.csv', index=False)
print('Saved MI ranking to artifacts/feature_importance_mi.csv')

## 7. Production Notes

Các feature trên sẽ được **port sang PySpark** trong `glue_jobs/silver_to_ml_features.py`:

| Pandas | PySpark equivalent |
|---|---|
| `groupby().shift(n)` | `F.lag(col, n).over(Window.partitionBy('city').orderBy('measured_at'))` |
| `rolling(n).mean()` | `F.avg(col).over(w.rowsBetween(-n, -1))` |
| `interpolate(method='time')` | `F.last(..., ignorenulls=True).over(w.rowsBetween(-6, -1))` (approx forward-fill) |
| `pd.Timestamp.hour` | `F.hour('measured_at')` |

**Quan trọng**: thứ tự operations trong PySpark phải giống pandas để Glue output match notebook results.